EMBEDDINGS E WORD VECOTRS (WORD2VEC, GLOVE, TF EMBEDDING LAYER)

L'intellegenza artificiale ha imparato a comprendere la distanza tra due parole.
La parola mela e la parola pera sono tra loro vicine, ma per un computer, fino a pochi anni fa, erano solo numeri isolati.
Il inguaggio è trasformato in una mappa geografica ed il significato diventa geometria.
- dal one-hot ai vettori densi
- i pilastri tecnologici Word2Vec e GloVe
- la distanza semantica tra due concetti usando la metrica del coseno

L'embedding è il ponte gra il linguaggio umano e la matematica.
Una rete neurale sa fare soltante una cosa:
calcolare con i numeri
Se scrivo: ordine
la rete non può farci nulla, nemmeno se scrivo: ordine cliente
Per la rete sono solo caratteri

Prima idea (sbagliata): 
potremmo numerare le parole 1=ordine 2=cliente, ecc
una frase diventa 1 e 2
In realtà è un grosso errore 2>1 1+1=2, ma non ha alcun significato linguistico.
Non esiste una relazione matematica tra ordine (1) e cliente (2). il numero è soltanto un identificatore.

Seconda idea:
usiamo un vettore di 0 e un 1 per la parola
ordine = [1000]
cliente = [0010]
Questo metodo si chiama one-hot encoding
Non funziona bene
Immagina un vocabolario di 100.000 parole, ogni parola richiederebbe 100.000 numeri quasi tutti zero.
E' uno spreco enorme di memoria
Ma c'è un problema ancora più grave, tutte le parole sembrano ugualmente lontane.
Per il one-hot encoding ordine è distante quanto cliente, perchè tutti i vettori sono ortogonali. Le rete non capisce che ordine e cliente sono concetti collegati

Terza idea (embedding):
Ogni parola non è più rappresentata da un numero, nemmeno da un vettore pieno di zeri, ma da un vettore di pochi numeri reali
Ordine potrebbe diventare [0.12,-0.4,0.88,...], magari 100 numeri.
Cosa rappresentano questi numeri? nessuno lo sa direttamente, non rappresentano grammatica, non rappresentano genere, non rappresentano verbo, ecc la rete li impara durante l'addestramento.
Come nasce l'embedding: supponiamo di leggere milioni di frasi, la rete vede spesso la parola 'ordine' e spesso vicino a 'cliente', 'fornitore', 'spedizione', ecc. quindi modifica lentamente il vettore della parola.
Dopo milioni di esempi otteniamo qualcosa del tipo [0.12,-0.4,0.88,...]. Parole usate in contesti simili avranno vettori simili

Word2Vec
Nel 2013 Google pubblico Word2Vec
Fu una rivoluzione
Per la prima volta si riuscirono a costruire embedding molto significativi.
Prendi una frase del tipo: 'il cliente invia un ordine', nascondi una parola 'il cliente invia un ___' il modello deve indovinare
Oppure fai il contrario, dai una parola 'ordine' e chiedi: 'quali parole stanno vicino?'
Allenandosi milioni di volte impara gli embedding
Due modalità:
1. Due modalità: input:'il ciente invia un ___' output: 'ordine'
2. Skip-Gram: input 'ordine' output: 'cliente invia'
Enrambi producono embedding

GloVe
Poco dopo Stanford proposte GloVe
L'idea è diversa
Word2Vec guarda finestre locali
GloVe guarda anche le statistiche globali del corpus.
Per esempio conta quante volte 'ordine' compare con 'cliente' rispetto a 'spedizione'
Usa una grande matrice di co-occorrenza

Differenza
Word2Vec impara facendo previsione
Parole vicine, predizione, embedding
GloVe impare dalle frequenze globali
Statistiche del corpus, embedding
Entrambi i metodi producono vettori

Word2Vec e GloVe hanno avuto un ruolo enorme nella storia del NLP, ma oggi sono  usati molto meno nei sistemi basati su Transformer.
Sia Word2Vec che GloVe utilizzano embedding statici, la parola banca ha la stessa rappresentazione sia nella frase 'vedo in banca a prelevare' che nella frase 'mi siedo sulla banca del fiume'

In uno spazio a centinaio di dimensioni, la distanza Euclidea no nè sempre la scelta migliore. Preferiamo analizzare l'angolo tra i vettori per capire quanto siano simili.
Queta metrica ha un nome preciso: la distanza coseno. Se la distanza coseno è 1 indica vettori quasi sovrapposti, paroli molto simili. Se la distanza coseno è zero, le due parole non hanno nulla in comune. Questa metrica è immuna alla frequenza delle parole, una parola molto frequente ed una rara possono avere la stessa distanza coseno se hanno significati simili. Termini sinonimi avranno una somiglianza del coseno molto elevata.

I Transformer hanno introdotto gli embedding contestuali: la rappresentazione della parola cambia in base al contesto della frase. E' uno dei motivi principali per cui i modelli come BERT, GPT e QWeen hanno superato le tecniche precedenti.



In [ ]:
# somiglianza tra parole

#pip install sentence_transformers
import numpy as np
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine

# 1. CARICAMENTO DEL MODELLO DA HUGGING FACE
# Teoria: 'all-mpnet-base-v2' mappa le parole in uno spazio a 768 dimensioni.
# È stato addestrato per raggruppare concetti semanticamente simili.
model = SentenceTransformer('all-mpnet-base-v2', device='cuda')

# 2. GENERAZIONE DI EMBEDDING REALI
# Modello funziona meglio in inglese
words = ["king", "queen", "apple", "lemon", "computer"]

# Teoria: Il modello analizza il contesto e assegna coordinate precise a ogni termine.
# encode() restituisce un array NumPy per ogni parola.
vectors = model.encode(words)

# Creiamo un dizionario 
word_vectors = dict(zip(words, vectors))

print(f"Dimensione del vettore per 'king': {word_vectors['king'].shape[0]} dimensioni")



# 3. FUNZIONE DI SIMILARITÀ (Invariata, la matematica non cambia!)
def manual_cosine_similarity(v1, v2):
    """
    Calcola la somiglianza del coseno tra due vettori.
    Formula: (A . B) / (||A|| * ||B||)
    """
    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    return dot_product / (norm_v1 * norm_v2)

# 4. CONFRONTO SEMANTICO REALE
target = "king"
comparison = "queen"

# Calcolo manuale
sim_manual = manual_cosine_similarity(word_vectors[target], word_vectors[comparison])

# Calcolo professionale (1 - distanza_coseno)
sim_pro = 1 - cosine(word_vectors[target], word_vectors[comparison])



print("-" * 30)
print(f"Confronto: '{target}' vs '{comparison}'")
print(f"Somiglianza Coseno: {sim_pro:.4f}")

# 5. TEST CROSS-CATEGORIA (Fruits vs Royalty)
sim_fruit = 1 - cosine(word_vectors["apple"], word_vectors["lemon"])
sim_diff = 1 - cosine(word_vectors["king"], word_vectors["apple"])

print("-" * 30)
print(f"Somiglianza 'apple' vs 'lemon' (Stessa categoria): {sim_fruit:.4f}")
print(f"Somiglianza 'king' vs 'apple' (Categorie diverse): {sim_diff:.4f}")

ModuleNotFoundError: No module named 'sentence_transformers'